In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os
import warnings

warnings.filterwarnings('ignore')
data_dir = "../data"

files = {
    "BIST100": "bist100_weekly.csv",
    "VIX": "vix_weekly.csv",
    "GOLD_TL": "gold_tl_weekly.csv",
    "BOND_2Y": "bond_2y_weekly.csv"
}

def safe_parse_price(val):
    if pd.isna(val): return np.nan
    val = str(val).strip()
    # Harfleri ve gereksiz sembolleri temizle (sadece rakam, nokta, virgül)
    import re
    val = re.sub(r'[^\d.,]', '', val)
    if not val: return np.nan
    
    # Format Tespiti
    if '.' in val and ',' in val:
        if val.rfind(',') > val.rfind('.'): # Türkçe format: 1.234,56
            val = val.replace('.', '').replace(',', '.')
        else: # ABD formatı: 1,234.56
            val = val.replace(',', '')
    elif ',' in val: # Sadece virgül varsa ondalık kabul et (örn: 15,4)
        val = val.replace(',', '.')
        
    try:
        return float(val)
    except:
        return np.nan

def load_and_clean(file_name, col_name):
    path = os.path.join(data_dir, file_name)
    df = pd.read_csv(path, sep=None, engine='python')
    df = df.iloc[:, [0, 1]].copy()
    df.columns = ["Date", col_name]
    
    # Tarihleri Standartlaştırma
    aylar = {"Oca":"Jan", "Şub":"Feb", "Mar":"Mar", "Nis":"Apr", "May":"May", "Haz":"Jun",
             "Tem":"Jul", "Ağu":"Aug", "Eyl":"Sep", "Eki":"Oct", "Kas":"Nov", "Ara":"Dec"}
    for tr, en in aylar.items():
        df["Date"] = df["Date"].astype(str).str.replace(tr, en, regex=False)
        
    df["Date"] = pd.to_datetime(df["Date"], format='mixed', dayfirst=True, errors='coerce')
    df[col_name] = df[col_name].apply(safe_parse_price)
    
    df = df.dropna().set_index("Date").sort_index()
    print(f"{col_name:8} -> Satır: {len(df)} | Başlangıç: {df.index[0].date()} | Bitiş: {df.index[-1].date()}")
    return df

print("Veriler Yükleniyor ve Temizleniyor...\n" + "-"*55)
dfs = [load_and_clean(file, key) for key, file in files.items()]
df_merged = pd.concat(dfs, axis=1, sort=True)

print(f"\n-> Birleştirme Sonrası Toplam Satır: {len(df_merged)}")

# Haftalık (Cuma) hizalama ve SADECE tatilleri atlamak için (limit=2) ffill
df_merged = df_merged.resample('W-FRI').last()
df_merged = df_merged.ffill(limit=2).dropna()

print(f"-> Haftalık Hizalama ve Sınırlandırılmış Doldurma Sonrası: {len(df_merged)}")

# Makalenin Matematiksel Formülü: Log ve Birinci Fark
df_log_diff = np.log(df_merged).diff().dropna()
df_log_diff = df_log_diff[['GOLD_TL', 'VIX', 'BOND_2Y', 'BIST100']]

print(f"-> Logaritma ve Fark Sonrası Kalan Satır: {len(df_log_diff)}")
if len(df_log_diff) == 0:
    raise ValueError("HATA: Veri seti boş!")

# Kronolojik Bölme (Son 180 hafta Test, öncesi Train)
test_size = 180
train_size = len(df_log_diff) - test_size

train_raw = df_log_diff.iloc[:train_size]
test_raw = df_log_diff.iloc[train_size:]

# ÖNEMLİ (Claude'un 2. Uyarısı): Scaler (Ölçekleyici) SADECE eğitim setine fit edilmeli!
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = pd.DataFrame(scaler.fit_transform(train_raw), columns=train_raw.columns, index=train_raw.index)
# Test setine sadece 'transform' uygulanır (gelecekten veri sızması önlenir)
test_scaled = pd.DataFrame(scaler.transform(test_raw), columns=test_raw.columns, index=test_raw.index)

train_scaled.to_csv(os.path.join(data_dir, "processed_train.csv"))
test_scaled.to_csv(os.path.join(data_dir, "processed_test.csv"))

print(f"\n✅ VERİ İŞLEME BAŞARILI! (Düz Çizgi Hatası Çözüldü & Sızıntı Engellendi)")
print(f"Eğitim (Train): {train_scaled.shape} | Test: {test_scaled.shape}")

Veriler Yükleniyor ve Temizleniyor...
-------------------------------------------------------
BIST100  -> Satır: 731 | Başlangıç: 2010-01-03 | Bitiş: 2023-12-31
VIX      -> Satır: 731 | Başlangıç: 2010-01-03 | Bitiş: 2023-12-31
GOLD_TL  -> Satır: 731 | Başlangıç: 2010-01-03 | Bitiş: 2023-12-31
BOND_2Y  -> Satır: 731 | Başlangıç: 2010-01-03 | Bitiş: 2023-12-31

-> Birleştirme Sonrası Toplam Satır: 731
-> Haftalık Hizalama ve Sınırlandırılmış Doldurma Sonrası: 731
-> Logaritma ve Fark Sonrası Kalan Satır: 730

✅ VERİ İŞLEME BAŞARILI! (Düz Çizgi Hatası Çözüldü & Sızıntı Engellendi)
Eğitim (Train): (550, 4) | Test: (180, 4)
